## Configure local inputs

Set the following environment variables to your files or folders before launching Jupyter: `MOUSEGPT_KEYPOINTS_JSON`, `MOUSEGPT_VIDEO`. Paths may include spaces. Output variables select destinations; input variables select existing session data. Review the analysis parameters in the cells before running on a new experiment.


In [ ]:
import os
required_paths = ['MOUSEGPT_KEYPOINTS_JSON', 'MOUSEGPT_VIDEO']
missing_paths = [name for name in required_paths if not os.environ.get(name)]
if missing_paths:
    raise ValueError("Set these environment variables before running: " + ", ".join(missing_paths))


In [ ]:
import json
import numpy as np
from tqdm import tqdm
import cv2

from scipy.signal import savgol_filter

import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from matplotlib.gridspec import GridSpec

In [ ]:
import os
print('Loading keypoints json ……')

json_path = os.environ['MOUSEGPT_KEYPOINTS_JSON']
with open(json_path, 'r') as f:
    data = json.load(f)
    
frame_num = len(data)
print(f'total {frame_num} frames')

In [ ]:
ears_pos = []

for frame in tqdm(data, desc='loading kpts'):
    ears = np.array(frame['instances'][0]['keypoints'][0:2])
    ears_pos.append(ears)

ears_pos = np.array(ears_pos)
print(ears_pos.shape)

In [ ]:
ear_speed = []

for index in tqdm(range(1, frame_num), desc='cal speed'):
    speed = np.linalg.norm(ears_pos[index]-ears_pos[index-1], axis=1)
    ear_speed.append(speed)

ear_speed = np.array(ear_speed)
print(ear_speed.shape)
print(ear_speed[:5])

In [ ]:
def filter(speed):
    # Apply the Savitzky-Golay filter to the 3D keypoint data
    window_length = 40
    poly_order = 7

    # Ensure window length is odd
    if window_length % 2 == 0:
        window_length += 1

    filtered_data_3d = np.empty_like(speed)

    # Apply the filter for each dimension separately
    for ear in range(2):
            filtered_data_3d[:, ear] = savgol_filter(speed[:, ear], window_length, poly_order)
    return filtered_data_3d

filtered_ear_speed = filter(ear_speed)

In [ ]:

fig, ax = plt.subplots()

ax.set_xlabel('Frame')
ax.set_ylabel('Speed')

ax.set_ylim(0, max(filtered_ear_speed[:, 0]))

line, = ax.plot([], [], 'b-')

In [ ]:
import os

cap = cv2.VideoCapture(os.environ['MOUSEGPT_VIDEO'])


fps = cap.get(cv2.CAP_PROP_FPS)


output_width = int(1280)
output_height = int(1280)
output_fps = fps//2
output_filename = 'output_video_max_4.mp4'
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter(output_filename, fourcc, output_fps, (output_width, output_height))

In [ ]:
index = 0
while cap.isOpened():
    index += 1
    print(index, end='\r')

    if index % 2 == 0:
        ret = cap.grab()
        continue
    else:
        ret, frame = cap.read()
    if not ret:
        break
    

    end_index = min(frame_num-1, index + 2)
    start_index = end_index-600
    
    current_speed_data = np.zeros(600)
    speed_start_index = max(0, start_index)
    print(speed_start_index, end_index)
    current_speed_data[max(0, 600-(end_index-speed_start_index)):600] = filtered_ear_speed[speed_start_index:end_index, 0]

    ax.set_xlim(max(start_index, 0), max(end_index, 600))
    

    line.set_data(range(start_index, end_index), current_speed_data)

    fig.canvas.draw()
    plt.pause(0.001)
    

    plt_img = np.frombuffer(fig.canvas.tostring_rgb(), dtype=np.uint8)
    plt_img = plt_img.reshape(fig.canvas.get_width_height()[::-1] + (3,))
    plt_img = cv2.cvtColor(plt_img, cv2.COLOR_RGB2BGR)
    plt_img = cv2.resize(plt_img, (1280,480))


    combined_frame = np.vstack([plt_img, frame])

    out.write(combined_frame)

    # if index > 5000:
    #     break


cap.release()
out.release()

In [ ]:

cap.release()
out.release()